# YOLOv8 Flag Classification Model Training Notebook

This notebook trains a **YOLOv8 Small Image Classifier** (`yolov8s-cls`) to classify flags into 254 country/institution categories, plus a background/negative class.

## Step 1: Install Dependencies
We install the `ultralytics` package.

In [ ]:
!pip install -U ultralytics

## Step 2: Locate Dataset
We scan `/kaggle/input` to locate the classification dataset.

In [ ]:
import os
import glob

# Search for any train directory inside /kaggle/input
train_dirs = glob.glob("/kaggle/input/**/train", recursive=True)
if train_dirs:
    dataset_dir = os.path.dirname(os.path.abspath(train_dirs[0]))
    print(f"Found input classification dataset at: {dataset_dir}")
    print("Splits:", os.listdir(dataset_dir))
else:
    print("[ERROR] Classification dataset not found!")
    dataset_dir = ""


## Step 3: Initialize Model
We load the pre-trained classification model weights (`yolov8s-cls.pt`).

In [ ]:
import torch
from ultralytics import YOLO

cuda_available = torch.cuda.is_available()
print(f"GPU (CUDA) Available: {cuda_available}")
if cuda_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    
model = YOLO("yolov8s-cls.pt")
print("Loaded yolov8s-cls.pt base model.")

## Step 4: Train classification model
We train the model. We default to **15 epochs** with a batch size of **32** and image size of **128**.

In [ ]:
if dataset_dir:
    device_val = 0 if torch.cuda.is_available() else 'cpu'
    model.train(
        data=dataset_dir,
        epochs=15,
        imgsz=128,
        batch=32,
        device=device_val,
        workers=2,
        name="yolov8s_cls_flag",
        exist_ok=True
    )
else:
    print("Skipping training because dataset was not found.")

## Step 5: Compress Outputs for Download
We package the final weights (`best.pt`, `last.pt`) and logs into a single `.zip` file for easy download from Kaggle.

In [ ]:
import zipfile

results_dir = "/kaggle/working/runs/classify/yolov8s_cls_flag"
zip_path = "/kaggle/working/yolov8s_cls_flag_results.zip"

if os.path.exists(results_dir):
    print(f"Compressing results from {results_dir}...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(results_dir):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, results_dir))
    print(f"Saved package to: {zip_path}")
else:
    print("Error: Training runs directory not found.")